# Problem 1: RNN, LSTM, GRU on Sequences
Student: Nilesh Ranjan Mitra | ID: 2026aj05496
Hidden size (h): 64 | Seeds: 42, 123, 999

In [10]:
import sys
sys.path.append("modules")

import numpy as np
import matplotlib.pyplot as plt

from datasets import load_har, create_subject_split, compute_normalization_stats, normalize_data, HARDataset, build_dataloaders
from plotting import load_activity_labels, plot_class_balance

DATA_DIR = "../Data/har/UCI HAR Dataset/UCI HAR Dataset"
SEEDS = [42, 123, 999]
BATCH_SIZE = 64

## Step 2 — Build HAR Data Pipeline (P1.1)
Load raw signals/labels/subjects, verify shapes, build the subject-wise
split, compute normalization statistics, and save class-balance plots.

In [11]:
x_train_full, y_train_full, subjects_train = load_har(DATA_DIR, "train")
x_test, y_test, subjects_test = load_har(DATA_DIR, "test")

print("train:", x_train_full.shape, y_train_full.shape, subjects_train.shape)
print("test: ", x_test.shape, y_test.shape, subjects_test.shape)

Finished stacking HAR signals for split 'train'
Finished loading HAR dataset for split 'train'
Finished stacking HAR signals for split 'test'
Finished loading HAR dataset for split 'test'
train: (7352, 128, 9) (7352,) (7352,)
test:  (2947, 128, 9) (2947,) (2947,)


In [12]:
# Shape checks against the assignment's documented sizes
assert x_train_full.shape == (7352, 128, 9), f"train shape wrong: {x_train_full.shape}"
assert x_test.shape == (2947, 128, 9), f"test shape wrong: {x_test.shape}"
assert y_train_full.shape == (7352,)
assert y_test.shape == (2947,)
assert subjects_train.shape == (7352,)
assert subjects_test.shape == (2947,)

# Label range checks (0-5 after the -1 shift in load_har)
assert y_train_full.min() == 0 and y_train_full.max() == 5
assert y_test.min() == 0 and y_test.max() == 5
assert len(np.unique(y_train_full)) == 6, "expected all 6 classes present in train"
assert len(np.unique(y_test)) == 6, "expected all 6 classes present in test"

# Subject counts and disjointness
unique_train_subjects = np.unique(subjects_train)
unique_test_subjects = np.unique(subjects_test)
assert len(unique_train_subjects) == 21, f"expected 21 train subjects, got {len(unique_train_subjects)}"
assert len(unique_test_subjects) == 9, f"expected 9 test subjects, got {len(unique_test_subjects)}"
assert set(unique_train_subjects).isdisjoint(set(unique_test_subjects)), "train/test subjects overlap!"

# No NaNs/Infs
assert np.isfinite(x_train_full).all(), "NaN/Inf found in train signals"
assert np.isfinite(x_test).all(), "NaN/Inf found in test signals"

print("All HAR shape and label checks passed.")

All HAR shape and label checks passed.


In [13]:
# Per-seed subject split validation
for seed in SEEDS:
    train_mask, val_mask = create_subject_split(subjects_train, seed)
    val_subjects = np.unique(subjects_train[val_mask])
    assert len(val_subjects) == 4, f"seed {seed}: expected 4 val subjects, got {len(val_subjects)}"
    assert train_mask.sum() + val_mask.sum() == len(subjects_train)
    assert not (train_mask & val_mask).any(), f"seed {seed}: train/val masks overlap"
    print(f"seed {seed}: train={train_mask.sum()}, val={val_mask.sum()}, val_subjects={val_subjects}")

Finished creating subject split with seed '42'
seed 42: train=5952, val=1400, val_subjects=[ 3 16 22 23]
Finished creating subject split with seed '123'
seed 123: train=5896, val=1456, val_subjects=[ 1  3 19 21]
Finished creating subject split with seed '999'
seed 999: train=5890, val=1462, val_subjects=[ 6 23 28 30]


### Class balance
Plots for the full train set, test set, and the per-seed train/val
subject subsets. Saved under `figures/class_balance/` for review;
final report figures will be selected after Problem 1 is complete.

In [14]:
class_names = load_activity_labels(DATA_DIR)

plot_class_balance(
    y_train_full, class_names,
    "Class Balance - Train (Full)",
    "figures/class_balance/train_full.png",
)
plot_class_balance(
    y_test, class_names,
    "Class Balance - Test",
    "figures/class_balance/test.png",
)

for seed in SEEDS:
    train_mask, val_mask = create_subject_split(subjects_train, seed)
    plot_class_balance(
        y_train_full[train_mask], class_names,
        f"Class Balance - Train Subset (seed={seed})",
        f"figures/class_balance/train_subset_seed{seed}.png",
    )
    plot_class_balance(
        y_train_full[val_mask], class_names,
        f"Class Balance - Val Subset (seed={seed})",
        f"figures/class_balance/val_subset_seed{seed}.png",
    )

print("Class balance plots saved.")

Finished loading activity labels
Finished plotting class balance to 'figures/class_balance/train_full.png'
Finished plotting class balance to 'figures/class_balance/test.png'
Finished creating subject split with seed '42'
Finished plotting class balance to 'figures/class_balance/train_subset_seed42.png'
Finished plotting class balance to 'figures/class_balance/val_subset_seed42.png'
Finished creating subject split with seed '123'
Finished plotting class balance to 'figures/class_balance/train_subset_seed123.png'
Finished plotting class balance to 'figures/class_balance/val_subset_seed123.png'
Finished creating subject split with seed '999'
Finished plotting class balance to 'figures/class_balance/train_subset_seed999.png'
Finished plotting class balance to 'figures/class_balance/val_subset_seed999.png'
Class balance plots saved.


### Normalization and DataLoaders
Normalization statistics are computed from the training-subject subset
only, for each seed, then applied to that seed's val set and to test.

In [15]:
seed = SEEDS[0]
train_mask, val_mask = create_subject_split(subjects_train, seed)

x_train_subset = x_train_full[train_mask]
y_train_subset = y_train_full[train_mask]
x_val_subset = x_train_full[val_mask]
y_val_subset = y_train_full[val_mask]

mean, std = compute_normalization_stats(x_train_subset)

x_train_norm = normalize_data(x_train_subset, mean, std)
x_val_norm = normalize_data(x_val_subset, mean, std)
x_test_norm = normalize_data(x_test, mean, std)

train_ds = HARDataset(x_train_norm, y_train_subset)
val_ds = HARDataset(x_val_norm, y_val_subset)
test_ds = HARDataset(x_test_norm, y_test)

train_loader, val_loader, test_loader = build_dataloaders(train_ds, val_ds, test_ds, BATCH_SIZE)

xb, yb = next(iter(train_loader))
print("batch shapes:", xb.shape, yb.shape)

Finished creating subject split with seed '42'
Finished computing normalization stats
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  0.38717854 0.25981337
   0.41742373 0.39939222 0.34735426]]]
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  0.38717854 0.25981337
   0.41742373 0.39939222 0.34735426]]]
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  0.38717854 0.25981337
   0.41742373 0.39939222 0.34

## Step 3 — Build Adding-Problem Pipeline (P1.1)
Generate the synthetic adding-problem dataset for T = 50, 100, 200, 400
using the assignment's exact `make_adding` function, and verify
correctness with assert tests.

In [16]:
from datasets import make_adding, AddingDataset

def verify_adding_data(x: np.ndarray, y: np.ndarray, T: int) -> None:
    """Assert exactly two markers, one per half, and target = sum of marked values."""
    marker_counts = x[:, :, 1].sum(axis=1)
    assert np.all(marker_counts == 2), "every sequence must have exactly two markers"

    first_half_markers = x[:, :T // 2, 1].sum(axis=1)
    second_half_markers = x[:, T // 2:, 1].sum(axis=1)
    assert np.all(first_half_markers == 1), "exactly one marker must be in the first half"
    assert np.all(second_half_markers == 1), "exactly one marker must be in the second half"

    i1 = np.argmax(x[:, :T // 2, 1], axis=1)
    i2 = T // 2 + np.argmax(x[:, T // 2:, 1], axis=1)
    expected_y = x[np.arange(len(x)), i1, 0] + x[np.arange(len(x)), i2, 0]
    assert np.allclose(y, expected_y, atol=1e-6), "target must equal sum of marked values"

In [17]:
T_VALUES = [50, 100, 200, 400]
N_TRAIN, N_VAL, N_TEST = 10_000, 1_000, 1_000
SEED_TRAIN, SEED_VAL, SEED_TEST = 0, 1, 2

adding_data = {}
for T in T_VALUES:
    x_train, y_train = make_adding(N_TRAIN, T, SEED_TRAIN)
    x_val, y_val = make_adding(N_VAL, T, SEED_VAL)
    x_test, y_test = make_adding(N_TEST, T, SEED_TEST)

    adding_data[T] = {
        "train": (x_train, y_train),
        "val": (x_val, y_val),
        "test": (x_test, y_test)
    }

print(f"T={T}: train={x_train.shape}, val={x_val.shape}, test={x_test.shape}")


for T in T_VALUES:
    for split in ["train", "val", "test"]:
        x, y = adding_data[T][split]
        verify_adding_data(x, y, T)
    print(f"T={T}: all checks passed (train/val/test)")

print("All adding-problem data verified successfully.")

Finished generating adding-problem data for T=50, n=10000, seed=0
Finished generating adding-problem data for T=50, n=1000, seed=1
Finished generating adding-problem data for T=50, n=1000, seed=2
Finished generating adding-problem data for T=100, n=10000, seed=0
Finished generating adding-problem data for T=100, n=1000, seed=1
Finished generating adding-problem data for T=100, n=1000, seed=2
Finished generating adding-problem data for T=200, n=10000, seed=0
Finished generating adding-problem data for T=200, n=1000, seed=1
Finished generating adding-problem data for T=200, n=1000, seed=2
Finished generating adding-problem data for T=400, n=10000, seed=0
Finished generating adding-problem data for T=400, n=1000, seed=1
Finished generating adding-problem data for T=400, n=1000, seed=2
T=400: train=(10000, 400, 2), val=(1000, 400, 2), test=(1000, 400, 2)
T=50: all checks passed (train/val/test)
T=100: all checks passed (train/val/test)
T=200: all checks passed (train/val/test)
T=400: all c

## Step 4 — Implement Custom Cells (P1.2)
Implement MyRNNCell, MyLSTMCell and MyGRUCell from matrix operations,
and verify their forward-pass correctness (shapes, finiteness,
determinism) before comparing them against PyTorch's library layers
in Step 5.

In [19]:
from models import MyRNNCell, MyLSTMCell, MyGRUCell
import torch

D = 9    # HAR channel count
H = 64   # your chosen hidden size
BATCH = 8
torch.manual_seed(0)
x_dummy = torch.randn(BATCH, D)
h0 = torch.zeros(BATCH, H)
c0 = torch.zeros(BATCH, H)

In [20]:
# Test MyRNN forward pass
rnn_cell = MyRNNCell(D, H)
h1 = rnn_cell(x_dummy, h0)

assert h1.shape == (BATCH, H), f"unexpected shape: {h1.shape}"
assert torch.isfinite(h1).all(), "RNN output contains NaN/Inf"

h1_again = rnn_cell(x_dummy, h0)
assert torch.equal(h1, h1_again), "RNN forward pass is not deterministic"

print("MyRNNCell forward pass OK:", h1.shape)

MyRNNCell forward pass OK: torch.Size([8, 64])


In [21]:
# Test MyLSTM forward pass
lstm_cell = MyLSTMCell(D, H)
h1, c1 = lstm_cell(x_dummy, (h0, c0))

assert h1.shape == (BATCH, H) and c1.shape == (BATCH, H), f"unexpected shapes: {h1.shape}, {c1.shape}"
assert torch.isfinite(h1).all() and torch.isfinite(c1).all(), "LSTM output contains NaN/Inf"

h1_again, c1_again = lstm_cell(x_dummy, (h0, c0))
assert torch.equal(h1, h1_again) and torch.equal(c1, c1_again), "LSTM forward pass is not deterministic"

print("MyLSTMCell forward pass OK:", h1.shape, c1.shape)

MyLSTMCell forward pass OK: torch.Size([8, 64]) torch.Size([8, 64])


In [22]:
# Test MyGRU forward pass
gru_cell = MyGRUCell(D, H)
h1 = gru_cell(x_dummy, h0)

assert h1.shape == (BATCH, H), f"unexpected shape: {h1.shape}"
assert torch.isfinite(h1).all(), "GRU output contains NaN/Inf"

h1_again = gru_cell(x_dummy, h0)
assert torch.equal(h1, h1_again), "GRU forward pass is not deterministic"

print("MyGRUCell forward pass OK:", h1.shape)

MyGRUCell forward pass OK: torch.Size([8, 64])


In [24]:
# Count the number of parameters for my chosen hidden size H (which will go in the Report file for reference)
import torch.nn as nn
def count_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters())

print("MyRNNCell params:", count_params(rnn_cell))   # expect 4*H + H*D + H*H  = 2H + HD + HH ... see Step 5 derivation
print("MyLSTMCell params:", count_params(lstm_cell))
print("MyGRUCell params:", count_params(gru_cell))

MyRNNCell params: 4800
MyLSTMCell params: 19200
MyGRUCell params: 14400


## Step 5 — Verify Against PyTorch (P1.2)
Copy weights from nn.RNN/nn.LSTM/nn.GRU into the custom cells, run
both over the same random sequence, and confirm the outputs match
within 1e-5. Then derive and reconcile parameter-count formulas.

In [ ]:
SEQ_LEN = 20  # arbitrary; any length works since we loop per time step

torch.manual_seed(0)
lib_rnn = nn.RNN(input_size=D, hidden_size=H, num_layers=1, batch_first=True)
lib_lstm = nn.LSTM(input_size=D, hidden_size=H, num_layers=1, batch_first=True)
lib_gru = nn.GRU(input_size=D, hidden_size=H, num_layers=1, batch_first=True)


def copy_weights(custom_cell: nn.Module, library_layer: nn.Module) -> None:
    """Copy layer-0 weights from a PyTorch RNN/LSTM/GRU into a matching custom cell.

    Works for all three because PyTorch names its single-layer parameters
    weight_ih_l0 / weight_hh_l0 / bias_ih_l0 / bias_hh_l0 regardless of
    gate count, and the gate concatenation order already matches ours.
    """
    with torch.no_grad():
        custom_cell.weight_ih.copy_(library_layer.weight_ih_l0)
        custom_cell.weight_hh.copy_(library_layer.weight_hh_l0)
        custom_cell.bias_ih.copy_(library_layer.bias_ih_l0)
        custom_cell.bias_hh.copy_(library_layer.bias_hh_l0)


copy_weights(rnn_cell, lib_rnn)
copy_weights(lstm_cell, lib_lstm)
copy_weights(gru_cell, lib_gru)
print("Weights copied into all three custom cells.")

In [ ]:
# Comparing over a random sequence for the custom RNN against the library RNN
# This ensures that our custom implementation produces the same outputs as PyTorch's built-in RNN for the same inputs and weights.
x_seq = torch.randn(BATCH, SEQ_LEN, D)  # (batch, seq_len, input_size)

lib_out, _ = lib_rnn(x_seq)  # (batch, seq_len, H)

h_t = torch.zeros(BATCH, H)
custom_outputs = []
for t in range(SEQ_LEN):
    h_t = rnn_cell(x_seq[:, t, :], h_t)
    custom_outputs.append(h_t)
custom_out = torch.stack(custom_outputs, dim=1)  # (batch, seq_len, H)

rnn_max_err = (custom_out - lib_out).abs().max().item()
assert rnn_max_err < 1e-5, f"RNN mismatch: max error {rnn_max_err}"
print(f"RNN max abs error: {rnn_max_err:.2e}")

In [ ]:
# Comparing over a random sequence for the custom LSTM against the library LSTM
# This ensures that our custom implementation produces the same outputs as PyTorch's built-in LSTM for the same inputs and weights.
lib_out, _ = lib_lstm(x_seq)  # (batch, seq_len, H)

h_t = torch.zeros(BATCH, H)
c_t = torch.zeros(BATCH, H)
custom_outputs = []
for t in range(SEQ_LEN):
    h_t, c_t = lstm_cell(x_seq[:, t, :], (h_t, c_t))
    custom_outputs.append(h_t)
custom_out = torch.stack(custom_outputs, dim=1)

lstm_max_err = (custom_out - lib_out).abs().max().item()
assert lstm_max_err < 1e-5, f"LSTM mismatch: max error {lstm_max_err}"
print(f"LSTM max abs error: {lstm_max_err:.2e}")

In [ ]:
# Comparing over a random sequence for the custom GRU against the library GRU
# This ensures that our custom implementation produces the same outputs as PyTorch's built-in GRU for the same inputs and weights.
lib_out, _ = lib_gru(x_seq)  # (batch, seq_len, H)

h_t = torch.zeros(BATCH, H)
custom_outputs = []
for t in range(SEQ_LEN):
    h_t = gru_cell(x_seq[:, t, :], h_t)
    custom_outputs.append(h_t)
custom_out = torch.stack(custom_outputs, dim=1)

gru_max_err = (custom_out - lib_out).abs().max().item()
assert gru_max_err < 1e-5, f"GRU mismatch: max error {gru_max_err}"
print(f"GRU max abs error: {gru_max_err:.2e}")

In [ ]:
# Building the verification table for custom RNN, LSTM, and GRU cells against PyTorch's built-in implementations
# This table will summarize the number of parameters (both actual and minimal) and the maximum absolute error for each custom cell.
# Also this will server as evidence in the final report 
import pandas as pd

def actual_params(gates: int, d: int, h: int) -> int:
    return gates * (h * d + h * h + 2 * h)

def minimal_params(gates: int, d: int, h: int) -> int:
    return gates * (h * d + h * h + h)

gate_counts = {"RNN": 1, "LSTM": 4, "GRU": 3}
max_errors = {"RNN": rnn_max_err, "LSTM": lstm_max_err, "GRU": gru_max_err}
custom_cells = {"RNN": rnn_cell, "LSTM": lstm_cell, "GRU": gru_cell}

rows = []
for name, gates in gate_counts.items():
    rows.append({
        "Model": name,
        "Gates": gates,
        "Actual params (formula)": actual_params(gates, D, H),
        "Actual params (code)": count_params(custom_cells[name]),
        "Minimal params (hand-derived)": minimal_params(gates, D, H),
        "Redundant bias params": gates * H,
        "Max abs error vs PyTorch": f"{max_errors[name]:.2e}",
    })

verification_table = pd.DataFrame(rows)
verification_table